### First look at the data

In [1]:
import pandas as pd

In [5]:
sheets = ["Year 2009-2010", "Year 2010-2011"]
df_list = []

for s in sheets:
    temp = pd.read_excel("../data/raw/online_retail.xlsx", sheet_name=s)
    temp["source_sheet"] = s   # keep track of which year it came from
    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

print(df.shape)
print(df.info())
print(df.isnull().sum())
df.head()

(1067371, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 9 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   Invoice       1067371 non-null  object        
 1   StockCode     1067371 non-null  object        
 2   Description   1062989 non-null  object        
 3   Quantity      1067371 non-null  int64         
 4   InvoiceDate   1067371 non-null  datetime64[ns]
 5   Price         1067371 non-null  float64       
 6   Customer ID   824364 non-null   float64       
 7   Country       1067371 non-null  object        
 8   source_sheet  1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(5)
memory usage: 73.3+ MB
None
Invoice              0
StockCode            0
Description       4382
Quantity             0
InvoiceDate          0
Price                0
Customer ID     243007
Country              0
source_sheet         0
dtype: in

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010


In [6]:
# Standardize column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Remove rows with missing Customer ID (can't tie sales to a customer)
df = df.dropna(subset=["customer_id"])

# Remove cancelled invoices (Invoice starting with 'C')
df = df[~df["invoice"].astype(str).str.startswith("C")]

# Remove bad rows: negative/zero quantity or price
df = df[(df["quantity"] > 0) & (df["price"] > 0)]

# Drop rows with missing description
df = df.dropna(subset=["description"])

# Create total sales column (needed for demand forecasting later)
df["total_sales"] = df["quantity"] * df["price"]

# Reset index after all the filtering
df = df.reset_index(drop=True)

print(df.shape)
print(df.isnull().sum())
df.head()

(805549, 10)
invoice         0
stockcode       0
description     0
quantity        0
invoicedate     0
price           0
customer_id     0
country         0
source_sheet    0
total_sales     0
dtype: int64


,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country,source_sheet,total_sales
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010,30.0


In [8]:
print(df["quantity"].min(), df["price"].min())   # should both be > 0
print(df["invoicedate"].min(), df["invoicedate"].max())  # sanity check date range

1 0.001
2009-12-01 07:45:00 2011-12-09 12:50:00


### Pushing Data to MySQL

In [10]:
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

In [14]:
from urllib.parse import quote_plus
user = os.getenv("MYSQL_USER")
password = quote_plus(os.getenv("MYSQL_PASSWORD"))  # encodes special chars like @
host = os.getenv("MYSQL_HOST")
db = os.getenv("MYSQL_DB")

engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}/{db}")

In [15]:
df.to_sql("sales_clean", con=engine, if_exists="replace", index=False)
print("Pushed to MySQL successfully")

Pushed to MySQL successfully
